### 문항 1 무신사 상품 10페이지

In [ ]:
# 무신사 상품 페이지에서 상의 카테고리 제품을 10페이지 크롤링해주세요.

# 대상: https://www.musinsa.com/category/001/goods?gf=A

# 추출 필드: 브랜드명 / 제품명 / 원래가격 / 할인가격 / 리뷰 수 / 리뷰 점수

# Selenium 혹은 requests로 수집

# Selenium으로 수집 시

# time.sleep()으로 요소를 기다리지 말 것. WebDriverWait + expected_conditions를 사용할 것

# headless 모드로 실행하고 창 크기를 명시할 것

# try / finally로 드라이버가 반드시 종료되게 할 것

# 결과를 musinsa.csv로 저장할 것

In [29]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd

In [24]:
URL = 'https://api.musinsa.com/api2/dp/v2/plp/goods'

param_str = 'gf=A&sortCode=POPULAR&category=001&size=60&testGroup=&caller=CATEGORY&page=2&hmacId=b2a9733ff4d74cefaa29f9ce9455f2bd643f1884def9325500a6e5b2f4b6e780&seen=61'
param =  [p.split('=') for p in param_str.split('&')]
param = {k: v for k, v in param}
param = {'gf': 'A',
 'sortCode': 'POPULAR',
 'category': '001',
 'size': '60',
 'caller': 'CATEGORY',
 'page': '1',
 'seen': '61'}

In [ ]:
res =requests.get(URL, params={**param})
res.json()

In [19]:
res.json()['data']['list'][0].get('brandName', '')

'나이스고스트클럽'

In [64]:
### 최적화
URL = 'https://api.musinsa.com/api2/dp/v2/plp/goods'

PARAMS = {'gf': 'A',
 'sortCode': 'POPULAR',
 'category': '001',
 'size': '60',
 'caller': 'CATEGORY',
 'page': '1',
 'seen': '61'}

def fetch(page, nextPageUrl=None):
    if nextPageUrl:
        res = requests.get(nextPageUrl)
    else: 
        res = requests.get(URL, params={**PARAMS, 'page':page})

    res.raise_for_status
    return res.json()

def parse(goods):
    list = []
    for item in goods['data']['list']:
        list.append({
            '브랜드명': item.get('brandName', ''),
            '제품명': item.get('goodsName', ''),
            '원래가격':item.get('normalPrice', ''),
            '할인가격':item.get('finalPrice', ''),
            '리뷰 수':item.get('reviewCount',''),
            '리뷰 점수':item.get('reviewScore', ''),
        })
    nextPageUrl = goods['data']['pagination']['nextPageUrl'] 
    return list, nextPageUrl

nextPageUrl = None
result = []
for page in range(1,11):
    list, nextPageUrl = parse(fetch(page, nextPageUrl))
    result.extend(list)
    print(f'{page}페이지 수집, 누적{len(result)}건')
    time.sleep(0.7)
df = pd.DataFrame(result).to_csv('musinsa.csv', index=False, encoding='utf-8-sig')

1페이지 수집, 누적60건
2페이지 수집, 누적120건
3페이지 수집, 누적180건
4페이지 수집, 누적240건
5페이지 수집, 누적300건
6페이지 수집, 누적360건
7페이지 수집, 누적420건
8페이지 수집, 누적480건
9페이지 수집, 누적540건
10페이지 수집, 누적600건


### 문항 2 로켓펀치 채용공고 10페이지

In [33]:
# 로켓펀치 채용 페이지에서 채용공고를 10페이지 수집하시오.

# 대상: https://www.rocketpunch.com/jobs

# 추출 필드: 기업명 / 공고명 / 요약 / 업무형태

# 결과를 rocketpunch.csv로 저장할 것

In [ ]:
URL = 'https://www.rocketpunch.com/api/proxy/jobs' 

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'x-rocket-api-version' : '3.0.0',
    'x-rocket-app-key': '077cd9d8-7237-4bee-b2d3-77690d163cca',
    'x-rocket-client-type': 'WEB',
    'x-rocket-device-type' : 'PC',
    'x-rocket-os-type': 'WIN',
}

# PARAM = {'sort': 'DATE_DESC'}

In [62]:
res = requests.get(URL, params={'sort': 'DATE_DESC'}, headers={**HEADERS})
res.raise_for_status
res.json()['items'][0]

{'jobId': 159214,
 'companyLogoUrl': 'https://image.rocketpunch.com/company/147791/realknowledge_logo_1747907586.png',
 'companyName': '에피소든',
 'companyPermalink': 'episoden',
 'title': 'Product Owner(PO)',
 'description': '글로벌 영어회화 서비스 PO 채용',
 'seniorities': ['미들', '시니어'],
 'workType': '상시 출근',
 'conditionMatches': [{'conditionType': 'JOB_CATEGORY', 'match': True},
  {'conditionType': 'SENIORITY', 'match': True},
  {'conditionType': 'COMPANY_SIZE', 'match': True},
  {'conditionType': 'WORK_TYPE', 'match': True}],
 'advertised': False,
 'originalLanguage': 'ko',
 'originalTitle': None,
 'translatedTitle': None,
 'translatedLanguage': None}

In [66]:
res.json()['pageToken'] #param에 넣어 다음 페이지 요청

'JnlscD-3ARYgn-UoUpL4qg=='

In [65]:
### 최적화
URL = 'https://www.rocketpunch.com/api/proxy/jobs' 

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'x-rocket-api-version' : '3.0.0',
    'x-rocket-app-key': '077cd9d8-7237-4bee-b2d3-77690d163cca',
    'x-rocket-client-type': 'WEB',
    'x-rocket-device-type' : 'PC',
    'x-rocket-os-type': 'WIN',
}

def fetch(pageToken):
    res = requests.get(URL, params={'sort': 'DATE_DESC', 'pageToken':pageToken}, headers={**HEADERS})
    res.raise_for_status
    return res.json()

def parse(jobs):
    list = []
    for item in jobs['items']:
        list.append({
            '기업명':item.get('companyName',''),
            '공고명':item.get('title',''),
            '요약':item.get('description',''),
            '업무형태':item.get('workType',''),
        })
    pageToken = jobs['pageToken']
    return list, pageToken

pageToken = None
result = []
for page in range(1,11):
    list, pageToken = parse(fetch(pageToken))
    result.extend(list)
    print(f"{page}페이지 · 누적 {len(result)}건")
    time.sleep(0.7)

df = pd.DataFrame(result).to_csv('rocketpunch.csv', index=False, encoding='utf-8-sig')


1페이지 · 누적 20건
2페이지 · 누적 40건
3페이지 · 누적 60건
4페이지 · 누적 80건
5페이지 · 누적 100건
6페이지 · 누적 120건
7페이지 · 누적 140건
8페이지 · 누적 160건
9페이지 · 누적 180건
10페이지 · 누적 200건
